# Import libraries

In [1]:
import pandas as pd
import numpy as np
import math
import random
import matplotlib.pyplot as plt 
import seaborn as sns

# Data Extraction and Pre-processing 

Load multi-year datasets from the US HUD
- Point-in-time (PIT) estimates of homelessness
- Housing inventory (HIC) data
- System performance measures

of about 400 Continuums of Care (CoCs) across the US.

In [4]:
# pip install pyxlsb

In [5]:
def extract_and_save_workbooks_as_csv(excel_file_name, extension, engine):
    excel_file = pd.ExcelFile(excel_file_name + extension)
    all_sheets = excel_file.sheet_names
    data_dict = {}

    for sheet in all_sheets:
        data_dict[sheet] = pd.read_excel(excel_file, sheet_name=sheet, engine = engine)

    for i in data_dict.keys():
        df_sheet = data_dict[i]
        df_sheet.to_csv(excel_file_name + "-" + i + '.csv')

In [ ]:
extract_and_save_workbooks_as_csv("System-Performance-Measures-Data", ".xlsx", None)
extract_and_save_workbooks_as_csv("2007-2023-HIC-Counts-by-CoC", ".xlsx", None)
extract_and_save_workbooks_as_csv("2007-2023-PIT-Counts-by-CoC", ".xlsb", "pyxlsb")

## Merge needed datasets

In [ ]:
for i in [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]:
    df = pd.read_csv("System-Performance-Measures-Data" + "-" + str(i) + ".csv")
    print(str(i) + ": " + str(len(df.columns)))

In [ ]:
for i in [2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]:
    df = pd.read_csv("2007-2023-HIC-Counts-by-CoC" + "-" + str(i) + ".csv")
    print(str(i) + ": " + str(len(df.columns)))

In [ ]:
for i in [2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]:
    df = pd.read_csv("2007-2023-PIT-Counts-by-CoC" + "-" + str(i) + ".csv")
    print(str(i) + ": " + str(len(df.columns)))

# Data Exploration

## EDA

### Line Chart

In [ ]:
main_df = spm_all.copy()
column = "Percent Returns in 24 mths (should include both the 6- and 12-month cohort) All"
complete_cocs = main_df.groupby('HUD CoC Number').filter(lambda x: x[column].notnull().all())['HUD CoC Number'].unique()
filtered_df = main_df[main_df['HUD CoC Number'].isin(complete_cocs)]
spm_all_mean = filtered_df.groupby('HUD CoC Number').agg({column: 'mean'}).reset_index()
extract = spm_all_mean.nsmallest(5, column)["HUD CoC Number"]
spm_all_extract = main_df[main_df["HUD CoC Number"].isin(extract)]
pivot_df = spm_all_extract.pivot_table(
    index='year', 
    columns='HUD CoC Number', 
    values= column
)
plt.figure(figsize=(8, 6))
for col in pivot_df.columns:
    plt.plot(pivot_df.index, pivot_df[col], marker='o', label=col)
plt.title("Percent Returns in 24 mths" + ' for Top 5 CoCs')
plt.xlabel('Year')
plt.ylabel("Percent Returns in 24 mths (All Housing Types)")
plt.legend(title='HUD CoC Number', bbox_to_anchor=(0.5, -0.15), loc='upper center', ncol = 5)
plt.grid(True)
plt.tight_layout()
plt.show()

## Break CoCs into small medium and large CoCs

In [ ]:
# Break down CoCs into small, medium and large CoCs before clustering
coc_focus = spm_all_1_complete_final["HUD CoC Number"].unique()
years1 = spm_all_1_complete_final["year"].unique()
column_names = ["CoC Number", "Sheltered Total Homeless", "year"]
df_pit_final = pd.DataFrame(columns=column_names)
for i in years1:
    df_pit = pd.read_csv("2007-2023-PIT-Counts-by-CoC" + "-" + str(i) + ".csv", 
                          usecols=['CoC Number', 'Sheltered Total Homeless'])
    df_pit = df_pit[df_pit["CoC Number"].isin(coc_focus)]
    df_pit["year"] = i
    df_pit_final = pd.concat([df_pit, df_pit_final], ignore_index=True)
df_pit_final.head()
df_pit_final_size = df_pit_final.groupby(["CoC Number"])["Sheltered Total Homeless"].mean().reset_index()
df_pit_final_size['CoC Size Category'] = pd.qcut(df_pit_final_size['Sheltered Total Homeless'],
    q=3, labels=['Small', 'Medium', 'Large']
)
df_pit_final_size
df_pit_final_size["CoC Size Category"].value_counts()

CoC Size Category
Small     76
Large     76
Medium    75
Name: count, dtype: int64

# Computing SHAP per cluster

In [ ]:
feature_names = X.columns

# For each cluster (class)
for i in range(7):
    print(f"Top 10 features for cluster {i}")

    # Get SHAP values for this cluster (class i)
    shap_vals_cluster = shap_values[:, :, i]  # shape: (n_samples, n_features)

    # Compute mean absolute SHAP values for each feature
    mean_abs_shap = np.mean(np.abs(shap_vals_cluster), axis=0)

    # Create a sorted Series
    top_features = pd.Series(mean_abs_shap, index=feature_names).sort_values(ascending=False).head(10)

    print(top_features)

In [ ]:
# Cluster label names mapping
cluster_names = {}

# Shortened feature labels
short_labels = {}

# Full SHAP values per cluster
top_features_dict = {}

# Convert to DataFrame
df = pd.DataFrame(top_features_dict).fillna(0)

# Apply short labels
df.index = df.index.to_series().replace(short_labels)

# Apply cluster names
df.columns = df.columns.map(cluster_names)

# Plot heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df, cmap="viridis", annot=True, fmt=".2f", cbar=True,
            mask=(df == 0), linewidths=0.5, linecolor='gray')
plt.title("Top 10 SHAP Feature Importances by Cluster")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()